# A4 · QC del cubo (M1–M5)

**Spec:** [`docs/spec_A4_codex_cube_qc.md`](../docs/spec_A4_codex_cube_qc.md)  |  **Bloque:** A · Reducción  |  **Run por defecto:** `ROXs12b_realigned`

Métricas de calidad del cubo: solución en λ (M1/M2), flujo absoluto (M3), STAT (M5).

| | |
|---|---|
| **Entrada** | `cube_telcorr.fits`, SKY_SPECTRUM, Gaia DR3 |
| **Salida (QC/productos)** | `stages/stage00q_qc.json` |
| **Consume aguas abajo** | D2/E1 (usan σ empírico), E3 (flujo) |


## Qué mide A4: las 5 métricas de calidad del cubo (M1–M5)

A4 no re-reduce: **verifica la calibración** del cubo con 5 métricas, cada una un aspecto distinto.

| Métrica | Qué mide | Resultado | Significado |
|---|---|---|---|
| **M1** | Exactitud de la solución de λ (offset vs airglow) | **green** (0.074 Å) | los λ del cubo están bien (~+3.4 km/s en Hα) |
| **M2** | LSF (ancho de la función de dispersión) | **yellow** (2.383 Å @Hα) | resolución espectral real; el NFM es ~10% más angosto que el nominal → se usa la medida en E1/E3/G2 |
| **M3** | Calibración de flujo absoluto (vs Gaia RP) | **green** (factor 0.973) | la escala de flujo casa con Gaia a ~3% |
| **M4** | Residuo de cielo (la `R` de A2) | **yellow** (R 0.547) | cielo bien restado (ver A2) |
| **M5** | Fiabilidad del STAT (varianza del cubo) | **red** (4.26×) | el STAT subestima el ruido ~4× → σ **siempre** empírico |

Las dos decisiones grandes de A4: **M3 cerrado** (flujo validado vs Gaia) y **M5 rojo** (STAT no sirve → regla *control = objeto*, [`docs/noise_model.md`](../docs/noise_model.md)).


## Cómo ejecutar de forma independiente

Etapa de **reducción**: la celda de abajo resuelve el comando real para **este objeto** a partir de su `chain.reduction_profile` y de su config, y puede lanzarlo. Son trabajos largos (ver coste), así que se lanzan en segundo plano con el log a la vista; el notebook no se bloquea.

Si algún dato no está declarado en el config del run, la celda lo dice y **no lanza** en vez de inventarse una ruta.

Comando histórico de referencia:

```bash
conda activate MUSE
# M1/M2 (LSF) desde el airglow cacheado:
python -m musepipe.qc.cube_qc m1m2-sky --sky-spectrum <SKY_SPECTRUM...> --qc-output <...>
# M3 (flujo absoluto vs Gaia RP, con growth-curve + truncación):
python -m musepipe.qc.cube_qc m3-flux --cube <cube_telcorr.fits> --run-id $RUN \
    --aperture-correction growth_curve --truncation-correction --qc-output <...>
```


In [ ]:
import os, sys
# Localiza la raíz del repo ascendiendo hasta encontrar `musepipe/` (robusto a
# la profundidad: funciona con el cwd en notebooks/<obj>/, en notebooks/ o en la
# raíz). Añade la raíz (para `import musepipe`) y notebooks/ (para `_nbcommon`).
_d = os.getcwd()
while _d != os.path.dirname(_d):
    if os.path.isdir(os.path.join(_d, 'musepipe')) and os.path.isdir(os.path.join(_d, 'notebooks')):
        break
    _d = os.path.dirname(_d)
_root = _d
for _p in (_root, os.path.join(_root, 'notebooks')):
    if _p not in sys.path:
        sys.path.insert(0, _p)
import _nbcommon as nb
RUN_ID = nb.resolve_run_id('ROXs12b_realigned')
print('run  =', RUN_ID)
print('root =', _root)
print('dir  =', nb.run_dir(RUN_ID))
print('QC   =', nb.provenance_line('stages/stage00q_qc.json', RUN_ID))


## Ejecutar o auditar


In [ ]:
cmd, target_run, missing = nb.launch_command('A4', RUN_ID)
print('run que ejecuta esta etapa:', target_run)
print('comando resuelto para este objeto:')
print('   ', cmd or '(sin plantilla)')
if missing:
    print()
    print('NO se puede lanzar: faltan datos en el config del run.')
    print('   sin resolver:', ', '.join(missing))
    print(f'   declara esas claves en runs/{target_run}/config/config.json')

RUN = False   # -> True para LANZAR (trabajo largo: revisa el coste arriba)

if RUN and not missing:
    import subprocess, time
    from pathlib import Path
    log = Path(nb.run_dir(target_run)) / 'logs' / f'a4_launch.log'
    log.parent.mkdir(parents=True, exist_ok=True)
    with open(log, 'w') as fh:
        proc = subprocess.Popen(cmd, shell=True, cwd=str(nb.project_root()),
                                stdout=fh, stderr=subprocess.STDOUT)
    print(f'lanzado en segundo plano (pid {proc.pid}); log -> {log}')
    print('sigue el progreso con:  !tail -f', log)
elif RUN:
    print('RUN=True pero hay datos sin resolver: no se lanza nada.')
else:
    print()
    print('Modo auditoría (RUN=False): abajo se carga el QC existente.')


## QC / resultados


In [ ]:
qc = nb.load_qc_optional('stages/stage00q_qc.json', RUN_ID)
nb.show(qc, keys=['m1_wavelength.status', 'm2_lsf.status', 'm3_flux.status', 'm4_sky.status', 'm5_stat.status'], title='A4')


## Resultados que llevaron a la conclusión

Resumen M1–M5 del `stage00q_qc.json` (estado + cifra de cabecera).


In [ ]:
if qc is None:
    print('(evidencia omitida: la etapa no se ha ejecutado para esta cadena)')
else:
    with nb.evidence_guard('A4', 'stages/stage00q_qc.json'):
        q = nb.load_qc('stages/stage00q_qc.json', RUN_ID)
        m1, m2, m3, m4, m5 = (q['m1_wavelength'], q['m2_lsf'], q['m3_flux'], q['m4_sky'], q['m5_stat'])
        rows = [
            ('M1 λ-solution', m1.get('status'), f"offset {m1.get('offset_median_A'):.3f} Å (±{m1.get('offset_err_A'):.3f}), {m1.get('n_lines')} líneas"),
            ('M2 LSF',        m2.get('status'), f"{m2.get('lsf_fwhm_at_halpha_A'):.3f} Å @Hα, dev vs nominal {m2.get('max_dev_vs_nominal_pct'):.1f}%"),
            ('M3 flujo abs',  m3.get('status'), f"factor {m3.get('flux_factor'):.3f} vs Gaia {m3.get('band')} (growth-curve r={m3.get('plateau_radius_px'):.0f})"),
            ('M4 cielo',      m4.get('status'), f"R = {m4.get('R')}"),
            ('M5 STAT',       m5.get('status'), f"factor spaxel {m5.get('factor_spaxel_median')}× (STAT subestima el ruido)"),
        ]
        for name, st, detail in rows:
            print(f'{name:15s} [{str(st):9s}] {detail}')


## Plot 1 — M2 LSF (medida vs nominal) y M3 growth-curve

Ambos desde el QC (baratos, sin cubo). **Izq:** la LSF medida del airglow (azul) cae bajo el nominal (gris) → el NFM es más angosto; línea en Hα = 2.383 Å. **Der:** el flujo en banda RP crece con el radio hasta el *plateau* (halo AO capturado) → `flux_factor = 0.973`.


In [ ]:
try:
    import numpy as np
    import matplotlib.pyplot as plt
    q = nb.load_qc('stages/stage00q_qc.json', RUN_ID)
    m2 = q['m2_lsf']; tab = m2['table_A_fwhm']
    w = np.array([r['wave_A'] for r in tab]); f = np.array([r['fwhm_A'] for r in tab])
    nomv = np.array([r['nominal_fwhm_A'] for r in tab])
    m3 = q['m3_flux']; gc = m3['growth_curve']
    gr = np.array([p['radius_px'] for p in gc]); gf = np.array([p['band_flux'] for p in gc])

    fig, (axL, axR) = plt.subplots(1, 2, figsize=(13, 4.2))
    axL.scatter(w, f, s=10, color='tab:blue', label='LSF medida (airglow)')
    axL.scatter(w, nomv, s=8, color='0.6', label='nominal (código)')
    axL.axvline(6563, color='tab:red', ls=':', label='Hα')
    hal = m2.get('lsf_fwhm_at_halpha_A')
    if hal: axL.axhline(hal, color='tab:red', ls='--', lw=1)
    axL.set_xlabel('λ [Å]'); axL.set_ylabel('FWHM LSF [Å]')
    axL.set_title(f"M2 · LSF medida vs nominal ({m2['status']}) · @Hα={hal:.3f} Å")
    axL.legend(fontsize=8)
    axR.plot(gr, gf, 'o-', color='tab:green')
    axR.axvline(m3['plateau_radius_px'], color='0.5', ls='--',
                label=f"plateau r={m3['plateau_radius_px']:.0f}px")
    axR.set_xlabel('radio de apertura [px]'); axR.set_ylabel('flujo en banda RP')
    axR.set_title(f"M3 · growth-curve → factor flujo={m3['flux_factor']:.3f} ({m3['status']})")
    axR.legend(fontsize=8); fig.tight_layout()
    outdir = nb.run_dir(RUN_ID) / 'plots' / 'a4_qc'; outdir.mkdir(parents=True, exist_ok=True)
    fig.savefig(outdir / 'm2_lsf_m3_growth.png', dpi=110)
    print('figura ->', outdir / 'm2_lsf_m3_growth.png'); plt.show()
except Exception as e:
    print('No se pudo generar el plot:', type(e).__name__, e)


## Plot 2 — M5: por qué el STAT no sirve (covarianza del remuestreo)

M5 mide que el STAT subestima el ruido **~4.26× por spaxel**. El QC solo guarda esa mediana, así que ilustro el **mecanismo** con la inflación espacial de G1 (almacenada): al sumar en cajas N×N la varianza real se infla frente a la suma ingenua de STAT (que asume píxeles independientes, =1) hasta ~19× en 5×5. Es la correlación introducida por el remuestreo del cubo → **σ siempre empírico, control = objeto**.

> Dependencia: este plot lee `stages/stage_g1_qc.json` (etapa G1); si G1 no ha corrido en el run, la celda degrada a un mensaje.


In [ ]:
try:
    import matplotlib.pyplot as plt
    q = nb.load_qc('stages/stage00q_qc.json', RUN_ID)
    m5 = q['m5_stat']
    g1 = nb.load_qc('stages/stage_g1_qc.json', RUN_ID)['covariance']
    infl = g1['spatial_inflation_by_box']
    boxes = sorted(infl, key=lambda k: int(k))
    xs = [f'{int(b)}×{int(b)}' for b in boxes]; vals = [infl[b] for b in boxes]

    fig, ax = plt.subplots(figsize=(8.5, 4.3))
    ax.bar(xs, vals, color='tab:orange', alpha=0.85)
    for i, v in enumerate(vals):
        ax.text(i, v + 0.3, f'{v:.1f}×', ha='center', fontsize=9)
    ax.axhline(1.0, color='tab:green', ls='--',
               label='STAT asume =1 (píxeles independientes)')
    ax.set_xlabel('caja de integración (N×N spaxels)')
    ax.set_ylabel('inflación varianza real / suma ingenua')
    ax.set_title(f"M5 [{m5['status']}]: STAT ~{m5['factor_spaxel_median']}× bajo por spaxel "
                 f"(+ covarianza del remuestreo en apertura)")
    ax.legend(fontsize=8); fig.tight_layout()
    outdir = nb.run_dir(RUN_ID) / 'plots' / 'a4_qc'; outdir.mkdir(parents=True, exist_ok=True)
    fig.savefig(outdir / 'm5_stat_inflation.png', dpi=110)
    print('figura ->', outdir / 'm5_stat_inflation.png'); plt.show()
except Exception as e:
    print('No se pudo generar el plot:', type(e).__name__, e)


## Decisiones y notas
- **M3 CERRADO (GREEN)**: flujo absoluto validado vs Gaia DR3 RP, factor 0.973 (~3%) tras growth-curve + truncación de cola.
- **M5 STAT en ROJO (inherente)**: el STAT subestima el ruido ~4.26× por covarianza del remuestreo → σ SIEMPRE empírico, control=objeto. Limitación aceptada en F1. · [`docs/noise_model.md`](../docs/noise_model.md)
- **M2 LSF@Hα = 2.383 Å medido** del airglow (NFM más angosta que el nominal 2.6); usada en E1/E3/G2.


## Conclusión (registrada)

**A4: cubo caracterizado; 2 verdes (M1, M3), 2 amarillos (M2, M4), 1 rojo inherente (M5).**

- **Fecha:** M1/M2 del airglow SKY_SPECTRUM y M3 vs Gaia, 2026-07-09.
- **M1 green:** offset 0.074 Å (solución de λ sana).
- **M2 yellow:** LSF 2.383 Å @Hα (NFM más angosto que nominal 2.6); es la LSF usada en E1/E3/G2.
- **M3 green:** flujo absoluto validado vs Gaia DR3 RP, factor 0.973 (~3%), con growth-curve (halo AO) + truncación de cola.
- **M4 yellow:** cielo bien restado (R=0.547, ver A2).
- **M5 red (inherente, no defecto):** STAT subestima ~4.26× por la covarianza del remuestreo → σ SIEMPRE empírico (control=objeto). Limitación aceptada en F1.
- **Impacto:** M5 fija la regla de ruido de toda la cadena (D2/E1/E3); M3 sostiene el flujo absoluto de E3.
